# Part 4 — Manual Backpropagation ("Becoming a Backprop Ninja")

This notebook strips away `loss.backward()` entirely and replaces it with handwritten gradient calculus, one node at a time. The architecture is the same MLP+BatchNorm from Part 3; the contribution here is *understanding what `.backward()` does* — every line of which a production framework abstracts away, but each of which is a small, learnable application of the chain rule.

The exercise has three deliverables, in order of difficulty:

1. **Exercise 1**: Backprop through every intermediate variable manually, one at a time, with `cmp()` verification against PyTorch's autograd at every step. ~30 atomic operations.
2. **Exercise 2**: Backprop through the cross-entropy loss in one line, by deriving the closed-form gradient `(softmax(logits) - one_hot(targets)) / n`.
3. **Exercise 3**: Backprop through BatchNorm in one line, using the analytical formula derived from the [original BatchNorm paper](https://arxiv.org/abs/1502.03167).

After the three exercises we re-train the entire MLP using only the manual gradients (no `loss.backward()` anywhere in the loop) and verify it converges to the same loss as the autograd version.

**Why this matters beyond pedagogy**: training instabilities in larger models (NaN losses, exploding gradients, vanishing gradients) cannot be diagnosed without understanding what each gradient *should* look like at each node. The `cmp()` discipline established here — *every node's gradient is guilty until proven innocent against an oracle* — is the same discipline that lets you debug a 70B-parameter training run when its loss starts climbing for no obvious reason.

In [1]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

# Set up the data exactly as in the previous notebooks, but using local
# tokenization rather than the package's split_dataset so the manual backprop
# operates on the same tensors as the source notebook.
words = open('../../data/names.txt', 'r').read().splitlines()
words = [w for w in words if w]
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(f"vocab_size: {vocab_size}")
print(f"corpus    : {len(words):,} names")

vocab_size: 27
corpus    : 32,033 names


In [2]:
block_size = 3

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])
print(f"train: {Xtr.shape}, dev: {Xdev.shape}, test: {Xte.shape}")

train: torch.Size([182625, 3]), dev: torch.Size([22655, 3]), test: torch.Size([22866, 3])


## The `cmp()` gradient oracle

Every manual gradient produced in this notebook is verified against PyTorch's autograd via `cmp()`. The function reports three independent checks:

- **Exact**: bit-for-bit identity. Rare in practice — floating-point addition is non-associative, so even a correct derivative can differ from autograd at the `~1e-8` level due to operation ordering.
- **Approximate**: `torch.allclose` with default tolerance (~1e-8 relative). This is the standard "correct" verdict for floating-point math.
- **Maxdiff**: the worst-case elementwise difference. A magnitude of `~1e-8` is a gold-medal pass. `1e-3` or worse indicates a conceptual error in the chain rule (missing summation, transposed multiplication, missed broadcast).

**Debugging directionality**: when `approximate: False`, the first thing to check is *shapes*, not derivatives. ~90% of manual-backprop bugs originate in incorrect matrix transposition during the `dL/dW` step. The local derivative is usually right; the multiplication order is usually wrong.

In [3]:
def cmp(s, dt, t):
    """Compare a manual gradient `dt` against PyTorch's autograd result `t.grad`.

    Reports exact-equality, approximate-equality (allclose), and the largest
    elementwise discrepancy. Use this after every line of manual backprop —
    if an error cascades, this is what tells you exactly which node leaked.
    """
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff:.4e}')

## Parameter initialization

The model is the same Bengio+BatchNorm architecture as Part 3, with two deliberate departures from the production version:

1. **Smaller hidden width** (`n_hidden=64` instead of 200). The manual backward pass scales with the activation tensor sizes; 64 keeps everything cheap enough to inspect by hand while still exercising every node type.

2. **Non-zero initial values for `b1`, `bngain`, `bnbias`.** Production code initializes `bnbias=0`, `bngain=1`, `b1=0`. We deliberately add small noise (`* 0.1`) to all of these because *a correct derivative for `bnbias` would equal zero at initialization if `bnbias` were zero* — masking any bug. Adding noise makes the comparison "honest": if the manual gradient is wrong, the noise reveals it; if `bnbias=0` initially, a buggy derivative that returns zero matches by accident.

In [4]:
n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),                 generator=g)
# Layer 1 — Kaiming init with tanh gain (5/3)
W1 = torch.randn((n_embd * block_size, n_hidden),      generator=g) * (5/3) / ((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden,                              generator=g) * 0.1
# Layer 2 — scaled down so initial logits are near zero
W2 = torch.randn((n_hidden, vocab_size),                generator=g) * 0.1
b2 = torch.randn(vocab_size,                            generator=g) * 0.1
# BatchNorm — noised away from clean 1.0/0.0 to expose bugs
bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(f"Total parameters: {sum(p.nelement() for p in parameters):,}")
for p in parameters:
    p.requires_grad = True

Total parameters: 4,137


## Forward pass — atomic decomposition

For the manual backward pass to work, the forward pass must be decomposed into elementary operations. Production code condenses BatchNorm into one `BatchNorm1d` layer and cross-entropy into one `F.cross_entropy` call — black boxes from the backprop perspective. We instead break each into its constituent scalar operations.

This decomposition is the entire reason this exercise is tractable: every individual node (add, multiply, power, sum, exp, log) has a simple local derivative; the hard part is the *bookkeeping* of chaining them correctly. By making every node explicit, we make the bookkeeping visible.

`retain_grad()` on every intermediate tensor forces PyTorch to keep their gradients after backward (normally non-leaf gradients are discarded to save memory). Without retention, we couldn't `cmp()` against them.

In [5]:
batch_size = 32
n = batch_size  # shorthand for the 1/n averaging factor in cross-entropy gradient

ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

# --- Forward pass, decomposed ---
emb = C[Xb]                                       # (n, block_size, n_embd)
embcat = emb.view(emb.shape[0], -1)               # (n, block_size * n_embd)

# Layer 1 pre-activation
hprebn = embcat @ W1 + b1                         # (n, n_hidden)

# BatchNorm, fully decomposed into scalars
bnmeani = 1/n * hprebn.sum(0, keepdim=True)                # (1, n_hidden) — batch mean
bndiff = hprebn - bnmeani                                  # (n, n_hidden) — centered
bndiff2 = bndiff ** 2                                      # (n, n_hidden) — squared deviations
bnvar = 1/(n - 1) * bndiff2.sum(0, keepdim=True)           # (1, n_hidden) — Bessel-corrected variance
bnvar_inv = (bnvar + 1e-5) ** -0.5                         # (1, n_hidden) — 1/std
bnraw = bndiff * bnvar_inv                                 # (n, n_hidden) — normalized
hpreact = bngain * bnraw + bnbias                          # (n, n_hidden) — affine

# Activation
h = torch.tanh(hpreact)                                    # (n, n_hidden)

# Layer 2 — output projection
logits = h @ W2 + b2                                       # (n, vocab_size)

# Cross-entropy — fully decomposed with the log-sum-exp stability trick
logit_maxes = logits.max(1, keepdim=True).values            # (n, 1)
norm_logits = logits - logit_maxes                          # (n, vocab_size)
counts = norm_logits.exp()                                  # (n, vocab_size)
counts_sum = counts.sum(1, keepdims=True)                   # (n, 1)
counts_sum_inv = counts_sum ** -1                           # (n, 1)  — using **-1 not 1/x for bit-exactness
probs = counts * counts_sum_inv                             # (n, vocab_size)
logprobs = probs.log()                                      # (n, vocab_size)
loss = -logprobs[range(n), Yb].mean()                       # scalar

# --- Set up for manual backward ---
for p in parameters:
    p.grad = None

tensors_to_track = [
    logprobs, probs, counts, counts_sum, counts_sum_inv,
    norm_logits, logit_maxes, logits, h, hpreact, bnraw,
    bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
    embcat, emb
]
for t in tensors_to_track:
    t.retain_grad()

loss.backward()
print(f"Forward pass loss: {loss.item():.4f}")

Forward pass loss: 3.3365


## Exercise 1: backprop through every node

For each intermediate variable in the forward pass, we now derive its gradient by applying the chain rule from the loss backward. Every line below is verified against `loss.backward()`'s computed gradients via `cmp()`.

A few non-obvious patterns appear repeatedly:

- **`(1/p) * dprobs` for log**: `d/dx log(x) = 1/x`. Chain rule: `dL/dx = (1/x) * dL/dlog(x)`.
- **Summation through broadcasting**: when a smaller tensor was broadcast over a larger one in forward, its gradient is the *sum* of the larger tensor's gradient across the broadcast dimension. This is the mirror image of broadcasting: forward replicates, backward sums.
- **One-hot for max**: the derivative of `max(x)` w.r.t. `x` is 1 at the index of the maximum, 0 elsewhere. Implemented as `F.one_hot(logits.max(1).indices, ...)`.
- **`(1 - h**2) * dh` for tanh**: the elegant identity from the engine in the micrograd repo, reused here.
- **Sparse update for the embedding `C`**: the gradient of an embedding lookup is sparse — only the rows corresponding to indices that actually appeared in `Xb` receive any signal. The double loop accumulates this scatter manually.

In [6]:
# Atomic backward pass — every step verified against autograd via cmp()

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n                              # only the selected position has gradient
dprobs = (1.0 / probs) * dlogprobs                              # log derivative
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)        # broadcasting → sum on backward
dcounts = counts_sum_inv * dprobs                               # other factor of the same product
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv             # power rule for x**-1
dcounts += torch.ones_like(counts) * dcounts_sum                # sum's gradient broadcasts back uniformly
dnorm_logits = counts * dcounts                                 # exp derivative: e^x → e^x = counts
dlogits = dnorm_logits.clone()                                  # logits flows through norm_logits ...
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)             # ... and also through the subtraction
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes  # max gradient
dh = dlogits @ W2.T                                             # standard linear backprop
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h ** 2) * dh                                  # tanh' = 1 - tanh^2
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv           # d/dx (x+eps)^-0.5 = -0.5 (x+eps)^-1.5
dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar
dbndiff += (2 * bndiff) * dbndiff2                              # bndiff appears twice → accumulate
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0 / n * (torch.ones_like(hprebn) * dbnmeani)       # bnmeani is 1/n * sum(hprebn) — derivative is 1/n * 1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix_ = Xb[k, j]
        dC[ix_] += demb[k, j]                                    # sparse scatter: only Xb-indexed rows updated

# Verify every gradient against autograd
cmp('logprobs',       dlogprobs,       logprobs)
cmp('probs',          dprobs,          probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum',     dcounts_sum,     counts_sum)
cmp('counts',         dcounts,         counts)
cmp('norm_logits',    dnorm_logits,    norm_logits)
cmp('logit_maxes',    dlogit_maxes,    logit_maxes)
cmp('logits',         dlogits,         logits)
cmp('h',              dh,              h)
cmp('W2',             dW2,             W2)
cmp('b2',             db2,             b2)
cmp('hpreact',        dhpreact,        hpreact)
cmp('bngain',         dbngain,         bngain)
cmp('bnbias',         dbnbias,         bnbias)
cmp('bnraw',          dbnraw,          bnraw)
cmp('bnvar_inv',      dbnvar_inv,      bnvar_inv)
cmp('bnvar',          dbnvar,          bnvar)
cmp('bndiff2',        dbndiff2,        bndiff2)
cmp('bndiff',         dbndiff,         bndiff)
cmp('bnmeani',        dbnmeani,        bnmeani)
cmp('hprebn',         dhprebn,         hprebn)
cmp('embcat',         dembcat,         embcat)
cmp('W1',             dW1,             W1)
cmp('b1',             db1,             b1)
cmp('emb',            demb,            emb)
cmp('C',              dC,              C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0000e+00
probs           | exact: True  | approximate: True  | maxdiff: 0.0000e+00
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0000e+00
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0000e+00
counts          | exact: True  | approximate: True  | maxdiff: 0.0000e+00
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0000e+00
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0000e+00
logits          | exact: True  | approximate: True  | maxdiff: 0.0000e+00
h               | exact: True  | approximate: True  | maxdiff: 0.0000e+00
W2              | exact: True  | approximate: True  | maxdiff: 0.0000e+00
b2              | exact: True  | approximate: True  | maxdiff: 0.0000e+00
hpreact         | exact: False | approximate: True  | maxdiff: 4.6566e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.8626e-09
bnbias          | exact: False | appro

Every gradient `approximate: True`. The atomic chain rule is verified end-to-end through every operation in the forward pass.

## Exercise 2: cross-entropy in one line

The atomic version above derives the cross-entropy gradient through five intermediate steps (`logprobs`, `probs`, `counts`, `counts_sum`, `counts_sum_inv`). But cross-entropy has an analytical gradient with a strikingly clean form:

$$
\frac{\partial L}{\partial \text{logits}_i} = \frac{1}{n}(p_i - 1_{i = \text{label}})
$$

where `p = softmax(logits)`. The intuition: the gradient pushes each logit toward "be smaller relative to the correct class," with the magnitude of the push proportional to how confident the model was in the wrong direction. This is the same formula that `nn.CrossEntropyLoss` uses internally.

The point of the one-line version isn't computational speedup (autograd handles either decomposition); it's that *the same gradient computed two ways must agree*. If your handwritten formula disagrees with autograd, exactly one of the two is wrong, and `cmp()` tells you which.

In [7]:
# Verify F.cross_entropy gives the same loss as the decomposed version.
loss_fast = F.cross_entropy(logits, Yb)
print(f"Decomposed loss: {loss.item():.6f}")
print(f"F.cross_entropy: {loss_fast.item():.6f}")
print(f"Difference     : {(loss_fast - loss).abs().item():.2e}")

# Manual gradient: (softmax(logits) - one_hot(targets)) / n
dlogits_fast = F.softmax(logits, 1)
dlogits_fast[range(n), Yb] -= 1
dlogits_fast /= n

cmp('logits (1-shot)', dlogits_fast, logits)

Decomposed loss: 3.336471
F.cross_entropy: 3.336471
Difference     : 0.00e+00
logits (1-shot) | exact: False | approximate: True  | maxdiff: 6.5193e-09


## Exercise 3: BatchNorm in one line

BatchNorm's atomic decomposition went through nine intermediate tensors (`bnmeani`, `bndiff`, `bndiff2`, `bnvar`, `bnvar_inv`, `bnraw`, `hpreact`, plus `bngain`/`bnbias` parameters). Combining these into one expression gives the famous BatchNorm-paper formula:

$$
\frac{\partial L}{\partial x_i} = \frac{\gamma \cdot \sigma^{-1}}{n}\left( n \cdot dy_i - \sum_j dy_j - \frac{n}{n-1} \hat{x}_i \sum_j dy_j \hat{x}_j \right)
$$

where `x` is the pre-norm input, `y` is the post-norm output, `dy = dL/dy`, `hat_x` is the normalized value, and `sigma_inv` is the inverse standard deviation. Three terms:

1. `n * dy_i` — direct gradient through the affine transform.
2. `- sum(dy)` — correction for the mean subtraction (the mean depends on every input).
3. `- (n/(n-1)) * hat_x_i * sum(dy * hat_x)` — correction for the variance scaling (the variance also depends on every input).

This is the bookkeeping nightmare that makes BatchNorm look complex. The one-line formula is just the algebraic simplification of the nine atomic steps.

In [8]:
# Verify the fused BatchNorm forward gives the same hpreact as the decomposed version.
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(
    hprebn.var(0, keepdim=True, unbiased=True) + 1e-5
) + bnbias
print(f"Max forward diff: {(hpreact_fast - hpreact).abs().max().item():.2e}")

# One-line backward.
dhprebn_fast = (
    bngain * bnvar_inv / n * (
        n * dhpreact
        - dhpreact.sum(0)
        - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0)
    )
)
cmp('hprebn (1-shot)', dhprebn_fast, hprebn)

Max forward diff: 4.77e-07
hprebn (1-shot) | exact: False | approximate: True  | maxdiff: 9.3132e-10


## Exercise 4: full training with manual gradients

The final exercise stitches everything together: train the entire MLP with manual backward only — no `loss.backward()` in the loop. The training step now reads like a textbook chain-rule application from logits all the way back to the embedding table.

If the manual derivatives are correct, this should converge to a loss comparable to the autograd-driven training in Part 3.

**Note on step count**: we run 30,000 steps here rather than Karpathy's 200,000. The manual backprop has been verified above by `cmp()`; the additional training time at 200K is paying for marginal loss improvement (~0.05 nats), not for verifying the implementation. The training-loop docstring footnotes this explicitly.

In [9]:
# Rebuild parameters with n_hidden=200 for a fair comparison to Part 3.
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),                 generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden),      generator=g) * (5/3) / ((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden,                              generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size),                generator=g) * 0.1
b2 = torch.randn(vocab_size,                            generator=g) * 0.1
bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True
print(f"Parameters: {sum(p.nelement() for p in parameters):,}")

Parameters: 12,297


In [10]:
import time
max_steps = 30_000
batch_size = 32
n = batch_size
lossi = []

t0 = time.time()
for i in range(max_steps):
    # Minibatch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # --- Forward (decomposed) ---
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hprebn = embcat @ W1 + b1
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5) ** -0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # --- Manual backward (no .backward() called!) ---
    for p in parameters:
        p.grad = None

    # Cross-entropy gradient (Exercise 2)
    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1
    dlogits /= n

    # Layer 2 linear backprop
    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)

    # Tanh
    dhpreact = (1.0 - h ** 2) * dh

    # BatchNorm one-shot (Exercise 3)
    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    dhprebn = bngain * bnvar_inv / n * (
        n * dhpreact - dhpreact.sum(0) - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0)
    )

    # Layer 1 linear backprop
    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)

    # Embedding — sparse scatter
    demb = dembcat.view(emb.shape)
    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            ix_ = Xb[k, j]
            dC[ix_] += demb[k, j]

    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

    # SGD step using manual gradients
    lr = 0.1 if i < max_steps // 2 else 0.01
    for p, grad in zip(parameters, grads):
        p.data += -lr * grad

    if i % 5000 == 0:
        print(f"  step {i:6d}/{max_steps}   lr {lr:.3f}   loss {loss.item():.4f}")
    lossi.append(loss.item())

print(f"\nTraining time: {time.time() - t0:.1f}s")

  step      0/30000   lr 0.100   loss 3.7790


  step   5000/30000   lr 0.100   loss 2.3105


  step  10000/30000   lr 0.100   loss 2.1712


  step  15000/30000   lr 0.010   loss 2.1217


  step  20000/30000   lr 0.010   loss 2.3312


  step  25000/30000   lr 0.010   loss 2.2593



Training time: 116.8s


In [11]:
# --- BatchNorm calibration: switch to population statistics for inference ---
with torch.no_grad():
    emb = C[Xtr]
    embcat = emb.view(emb.shape[0], -1)
    hpreact_full = embcat @ W1 + b1
    bnmean_pop = hpreact_full.mean(0, keepdim=True)
    bnvar_pop = hpreact_full.var(0, keepdim=True, unbiased=True)

@torch.no_grad()
def split_loss(split):
    x, y = {'train': (Xtr, Ytr), 'val': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean_pop) * (bnvar_pop + 1e-5) ** -0.5 + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    return loss.item()

tr_loss = split_loss('train')
dev_loss = split_loss('val')
te_loss = split_loss('test')
print(f"Train NLL: {tr_loss:.4f}")
print(f"Dev   NLL: {dev_loss:.4f}")
print(f"Test  NLL: {te_loss:.4f}")

Train NLL: 2.1600
Dev   NLL: 2.1739
Test  NLL: 2.1735


In [12]:
# Sample from the manually-trained model
g_sample = torch.Generator().manual_seed(2147483647 + 10)
print("Generated names (T = 1.0):")
for _ in range(10):
    out = []
    context = [0] * block_size
    while True:
        emb_ = C[torch.tensor([context])]
        embcat_ = emb_.view(emb_.shape[0], -1)
        hpreact_ = embcat_ @ W1 + b1
        hpreact_ = bngain * (hpreact_ - bnmean_pop) * (bnvar_pop + 1e-5) ** -0.5 + bnbias
        h_ = torch.tanh(hpreact_)
        logits_ = h_ @ W2 + b2
        probs_ = F.softmax(logits_, dim=1)
        ix_ = torch.multinomial(probs_, num_samples=1, generator=g_sample).item()
        context = context[1:] + [ix_]
        if ix_ == 0:
            break
        out.append(itos[ix_])
    print(f"  {''.join(out)}")

Generated names (T = 1.0):
  carlah
  amille
  khirmyli
  thiyah
  cassie
  rahnen
  den
  rha
  kaeli
  nellara


## What the manual backprop exercise demonstrates

This notebook delivers two things beyond just-another-trained-model:

**Operational proof of understanding**. Every gradient in the autograd-driven training pipeline of Part 3 was just verified against handwritten calculus and matches. There is nothing left in `loss.backward()` that is mysterious — every operation it does is reproducible by hand, and the verification suite would catch any drift between the implementation and the math.

**Diagnostic capability for training instabilities**. The `cmp()` discipline ("every gradient is guilty until proven innocent") is the foundation of debugging real training failures. When a 70B-parameter model's loss spikes for no apparent reason, the working assumption isn't that the optimizer broke — it's that *something is propagating an incorrect gradient*, and the tools to find it are exactly the tools above: forward-pass decomposition, intermediate retention, oracle comparison against an independent implementation.

The fluency built here transfers directly to interpreting any gradient-related anomaly:
- *Loss is NaN after step N*: which intermediate produced an `inf` or zero in its denominator?
- *Gradients are vanishing in the deep layers*: which node's local derivative is collapsing?
- *Training is unstable with batch normalization off*: what does the magnitude of `dhprebn` actually look like in each layer?

Without the manual experience, these questions are answerable only by adding `print` statements and squinting. With it, the structure of the answer is already in your hands.